In [1]:
import torch
import torch.nn as nn
import math
import os
import cv2
import numpy as np
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
import pandas as pd

In [2]:
## Helper Functions

In [3]:
def train_epoch(model, train_loader, optimizer, criterion, epoch, device):
    """ Training a model for one epoch """
    
    loss_list = []
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader))
    for i, (images, labels) in progress_bar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Clear gradients w.r.t. parameters    )
        optimizer.zero_grad()
         
        # Forward pass to get output/logits
        outputs = model(images)
         
        # Calculate Loss: softmax --> cross entropy loss
        loss = criterion(outputs, labels)
        loss_list.append(loss.item())
         
        # Getting gradients w.r.t. parameters
        loss.backward()
         
        # Updating parameters
        optimizer.step()
        
        progress_bar.set_description(f"Epoch {epoch+1} Iter {i+1}: loss {loss.item():.5f}. ")
        
    mean_loss = np.mean(loss_list)
    return mean_loss, loss_list


@torch.no_grad()
def eval_model(model, eval_loader, criterion, device):
    """ Evaluating the model for either validation or test """
    correct = 0
    total = 0
    loss_list = []
    
    for images, labels in eval_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass only to get logits/output
        outputs = model(images)
                 
        loss = criterion(outputs, labels)
        loss_list.append(loss.item())
            
        # Get predictions from the maximum value
        preds = torch.argmax(outputs, dim=1)
        correct += len( torch.where(preds==labels)[0] )
        total += len(labels)
                 
    # Total correct predictions and loss
    accuracy = correct / total * 100
    loss = np.mean(loss_list)
    
    return accuracy, loss


def train_model(model, optimizer, scheduler, criterion, train_loader, valid_loader, num_epochs):
    """ Training a model for a given number of epochs"""
    
    train_loss = []
    val_loss =  []
    loss_iters = []
    valid_acc = []
    
    for epoch in range(num_epochs):
           
        # validation epoch
        model.eval()  # important for dropout and batch norms
        accuracy, loss = eval_model(
                    model=model, eval_loader=valid_loader,
                    criterion=criterion, device=device
            )
        valid_acc.append(accuracy)
        val_loss.append(loss)
        
        # training epoch
        model.train()  # important for dropout and batch norms
        mean_loss, cur_loss_iters = train_epoch(
                model=model, train_loader=train_loader, optimizer=optimizer,
                criterion=criterion, epoch=epoch, device=device
            )
        scheduler.step()
        train_loss.append(mean_loss)
        loss_iters = loss_iters + cur_loss_iters
        
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"    Train loss: {round(mean_loss, 5)}")
        print(f"    Valid loss: {round(loss, 5)}")
        print(f"    Accuracy: {accuracy}%")
        print("\n")
        save_model(
                model=model,
                optimizer=optimizer,
                epoch=epoch,
                stats={
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "loss_iters": loss_iters,
                    "valid_acc": valid_acc,
                }
            )
    
    print(f"Training completed")
    return train_loss, val_loss, loss_iters, valid_acc


def smooth(f, K=5):
    """ Smoothing a function using a low-pass filter (mean) of size K """
    kernel = np.ones(K) / K
    f = np.concatenate([f[:int(K//2)], f, f[int(-K//2):]])  # to account for boundaries
    smooth_f = np.convolve(f, kernel, mode="same")
    smooth_f = smooth_f[K//2: -K//2]  # removing boundary-fixes
    return smooth_f

def count_model_params(model):
    """ Counting the number of learnable parameters in a nn.Module """
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return num_params


def save_model(model, optimizer, epoch, stats):
    """ Saving model checkpoint """
    
    if(not os.path.exists("models")):
        os.makedirs("models")
    savepath = f"models/checkpoint_epoch_{epoch}.pth"

    if isinstance(model.lstm, nn.LSTM):
        recurrent_impl = "lstm"
    elif isinstance(model.lstm, nn.ModuleList):
        recurrent_impl = "lstm_cells"
    else:
        recurrent_impl = type(model.lstm).__name__

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'stats': stats,
        'model_meta': {
            'class_name': model.__class__.__name__,
            'recurrent_impl': recurrent_impl,
            'num_layers': model.num_layers,
            'hidden_dim': model.hidden_dim,
        }
    }, savepath)
    return


def _convert_recurrent_state_dict(source_state_dict, target_state_dict):
    """Translate between nn.LSTM and stacked nn.LSTMCell parameter names."""

    source_keys = set(source_state_dict.keys())
    target_keys = set(target_state_dict.keys())
    converted_state_dict = dict(source_state_dict)

    source_is_lstm = any(key.startswith("lstm.weight_ih_l") for key in source_keys)
    target_is_lstm = any(key.startswith("lstm.weight_ih_l") for key in target_keys)
    source_is_cells = any(key.startswith("lstm.0.weight_ih") for key in source_keys)
    target_is_cells = any(key.startswith("lstm.0.weight_ih") for key in target_keys)

    if source_is_lstm and target_is_cells:
        num_layers = sum(1 for key in source_keys if key.startswith("lstm.weight_ih_l"))
        for layer_idx in range(num_layers):
            converted_state_dict[f"lstm.{layer_idx}.weight_ih"] = converted_state_dict.pop(f"lstm.weight_ih_l{layer_idx}")
            converted_state_dict[f"lstm.{layer_idx}.weight_hh"] = converted_state_dict.pop(f"lstm.weight_hh_l{layer_idx}")
            converted_state_dict[f"lstm.{layer_idx}.bias_ih"] = converted_state_dict.pop(f"lstm.bias_ih_l{layer_idx}")
            converted_state_dict[f"lstm.{layer_idx}.bias_hh"] = converted_state_dict.pop(f"lstm.bias_hh_l{layer_idx}")
        return converted_state_dict, "lstm", "lstm_cells"

    if source_is_cells and target_is_lstm:
        layer_indices = sorted({int(key.split(".")[1]) for key in source_keys if key.startswith("lstm.")})
        for layer_idx in layer_indices:
            converted_state_dict[f"lstm.weight_ih_l{layer_idx}"] = converted_state_dict.pop(f"lstm.{layer_idx}.weight_ih")
            converted_state_dict[f"lstm.weight_hh_l{layer_idx}"] = converted_state_dict.pop(f"lstm.{layer_idx}.weight_hh")
            converted_state_dict[f"lstm.bias_ih_l{layer_idx}"] = converted_state_dict.pop(f"lstm.{layer_idx}.bias_ih")
            converted_state_dict[f"lstm.bias_hh_l{layer_idx}"] = converted_state_dict.pop(f"lstm.{layer_idx}.bias_hh")
        return converted_state_dict, "lstm_cells", "lstm"

    return converted_state_dict, None, None


def load_model(model, optimizer, savepath):
    """ Loading pretrained checkpoint """
    
    checkpoint = torch.load(savepath, weights_only=False)
    state_dict = checkpoint['model_state_dict']

    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        converted_state_dict, source_impl, target_impl = _convert_recurrent_state_dict(
            source_state_dict=state_dict,
            target_state_dict=model.state_dict(),
        )
        if source_impl is None:
            raise
        model.load_state_dict(converted_state_dict)
        print(f"Converted checkpoint recurrent weights from {source_impl} to {target_impl}.")

    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint["epoch"]
    stats = checkpoint["stats"]
    
    return model, optimizer, epoch, stats

In [4]:
class LSTMScratch(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        
        # Explicit weights for each gate
        # (Input + Hidden) * Output_size
        self.W_i = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_g = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, states=None):
        b_size, seq_len, _ = x.shape
        h_t = torch.zeros(b_size, self.hidden_size, device=x.device)
        c_t = torch.zeros(b_size, self.hidden_size, device=x.device)
        
        outputs = []
        
        for t in range(seq_len):
            x_t = x[:, t, :]
            # Combine input and hidden state into one vector for each step
            combined = torch.cat((x_t, h_t), dim=1)
            
            # Now compute each gate separately and clearly
            i_t = torch.sigmoid(self.W_i(combined))
            f_t = torch.sigmoid(self.W_f(combined))
            g_t = torch.tanh(self.W_g(combined))
            o_t = torch.sigmoid(self.W_o(combined))
            
            # Update States
            c_t = f_t * c_t + i_t * g_t
            h_t = o_t * torch.tanh(c_t)
            
            outputs.append(h_t)
            
        return torch.stack(outputs, dim=1), (h_t, c_t)

In [5]:
import torch
import torch.nn as nn

class ConvLSTMCellCustom(nn.Module):
    """
    A single cell of a ConvLSTM.
    Processes spatial data (channels, height, width) through 4 explicit gate convolutions.
    """
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        padding = kernel_size // 2
        
        # Defining 4 separate convolutional kernels for the 4 LSTM gates
        # Each kernel accepts the concatenation of Input and Hidden state
        self.conv_i = nn.Conv2d(in_channels + hidden_channels, hidden_channels, kernel_size, padding=padding)
        self.conv_f = nn.Conv2d(in_channels + hidden_channels, hidden_channels, kernel_size, padding=padding)
        self.conv_g = nn.Conv2d(in_channels + hidden_channels, hidden_channels, kernel_size, padding=padding)
        self.conv_o = nn.Conv2d(in_channels + hidden_channels, hidden_channels, kernel_size, padding=padding)

    def forward(self, x, states):
        h_prev, c_prev = states
        
        # Combine input and previous hidden state along the channel dimension
        combined = torch.cat([x, h_prev], dim=1)
        
        # Compute gates explicitly
        i_t = torch.sigmoid(self.conv_i(combined)) # Input Gate
        f_t = torch.sigmoid(self.conv_f(combined)) # Forget Gate
        g_t = torch.tanh(self.conv_g(combined))    # Candidate Gate
        o_t = torch.sigmoid(self.conv_o(combined)) # Output Gate
        
        # Update Cell State and Hidden State
        c_next = f_t * c_prev + i_t * g_t
        h_next = o_t * torch.tanh(c_next)
        
        return h_next, c_next

class ConvLSTMCustom(nn.Module):
    """
    The full ConvLSTM model that iterates over a sequence of images.
    """
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.cell = ConvLSTMCellCustom(in_channels, hidden_channels, kernel_size)
        self.hidden_channels = hidden_channels

    def forward(self, x):
        # x shape: (batch, time, channels, height, width)
        b, t, c, h, w = x.shape
        
        # Initialize hidden and cell states with zeros
        h_t = torch.zeros(b, self.hidden_channels, h, w, device=x.device)
        c_t = torch.zeros(b, self.hidden_channels, h, w, device=x.device)
        
        outputs = []
        for i in range(t):
            h_t, c_t = self.cell(x[:, i, ...], (h_t, c_t))
            outputs.append(h_t)
            
        # Stack output frames along the time dimension
        return torch.stack(outputs, dim=1)

#### Simple comparision
The main difference between the ConvLSTM and regural LSTM, is the way the gates are handles the LSTM gates are handled via a linear layer, while ConvLSTM is handled using Convulotional layer ... common sense.


The Fundamental Difference: ConnectivityStandard LSTM (Fully Connected): Treats the input as a "bag of features" (a 1D vector). When it calculates gates, it multiplies every input feature by every weight. It destroys the spatial grid of the image, meaning it has no inherent sense that pixel (10,10) is neighbors with pixel (10,11).ConvLSTM (Spatially Aware): Treats the input as a "spatial grid" (a 3D tensor: Channels * Height * Width). It uses convolutional kernels to calculate gates. Because a convolution only looks at a small local patch of pixels, it preserves the spatial structure of the data.

In [6]:
class KTHDataset(Dataset):
    def __init__(self, data_path, seq_len=10, img_size=(64, 64)):
        self.data_path = data_path
        self.seq_len = seq_len
        self.img_size = img_size
        self.samples = self._prepare_samples()
        self.labels_map = {"boxing": 0, "handclapping": 1, "handwaving": 2, 
                           "jogging": 3, "running": 4, "walking": 5}

    def _prepare_samples(self):
        """Scans directory and creates list of (video_path, label)"""
        samples = []
        for class_name in os.listdir(self.data_path):
            if class_name in self.labels_map:
                cls_dir = os.path.join(self.data_path, class_name)
                for video_file in os.listdir(cls_dir):
                    samples.append((os.path.join(cls_dir, video_file), self.labels_map[class_name]))
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        cap = cv2.VideoCapture(video_path)
        frames = []
        
        while len(frames) < self.seq_len:
            ret, frame = cap.read()
            if not ret: # If video ends, loop or pad
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue
            
            # Preprocess: Grayscale and Resize
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frame = cv2.resize(frame, self.img_size)
            frame = frame / 255.0 # Normalize
            frames.append(frame)
            
        cap.release()
        
        # Convert to tensor: (Time, Channel, Height, Width)
        video_tensor = torch.tensor(np.array(frames), dtype=torch.float32).unsqueeze(1)
        return video_tensor, label

In [8]:
# My Model

import torch
import torch.nn as nn

class SpatialEncoder(nn.Module):
    """CNN Encoder to extract spatial features."""
    def __init__(self, in_channels=1, out_features=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), # Output: 32x32
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), # Output: 16x16
            nn.Conv2d(128, out_features, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)) # Spatial preservation before flattening
        )
    def forward(self, x): return self.encoder(x)

class ActionClassifier(nn.Module):
    """Classifier using Conv -> AvgPooling -> Linear with Dropout."""
    def __init__(self, in_channels, num_classes=6, dropout_prob=0.5):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(p=dropout_prob), # Added Dropout here
            nn.Linear(in_channels, num_classes)
        )
    def forward(self, x): return self.classifier(x)
        
class HybridVideoModel(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.encoder = SpatialEncoder()
        self.lstm = nn.LSTM(input_size=128*4*4, hidden_size=256, batch_first=True)
        self.classifier = ActionClassifier(in_channels=128)

    def forward(self, x):
        # 1. x shape: (batch, time, channels, height, width)
        b, t, c, h, w = x.shape
        
        # 2. Reshape for the CNN Encoder
        x_reshaped = x.view(b * t, c, h, w)
        
        # 3. Spatial Encoding
        features = self.encoder(x_reshaped) # (b*t, 128, 4, 4)
        
        # 4. Flatten for the LSTM
        features = features.view(b, t, -1) # (b, t, 2048)
        
        lstm_out, _ = self.lstm(features)
        
        # 6. Prepare for Classifier: Extract last step and reshape
        last_hidden = lstm_out[:, -1, :].view(b, 128, 4, 4)
        
        return self.classifier(last_hidden)


This CNN-RNN hybrid processes $64 \times 64$ frames in three stages:Spatial Encoder: A 3-layer CNN compresses frames into a $4 \times 4$ feature grid.Recurrent Module (LSTM): Models temporal changes in this grid.Classifier: A Conv-pool-linear block aggregates the final state into one of six action labels.

In [12]:
def create_kth_splits(root_path):
    data = []
    actions = ["boxing", "handclapping", "handwaving", "jogging", "running", "walking"]

    print(os.getcwd())
    for action in actions:
        action_path = os.path.join(root_path, action)
        for video_file in os.listdir(action_path):
            if video_file.endswith(".avi"):
                # Pattern: personXX_action_...
                person_id = int(video_file.split('_')[0].replace('person', ''))
                data.append({
                    "path": os.path.join(action_path, video_file),
                    "action": action,
                    "person": person_id
                })
                
    df = pd.DataFrame(data)
    # KTH standard: Persons 1-16 for training, 17-25 for validation
    train_df = df[df['person'] <= 16]
    val_df = df[df['person'] > 16]
    
    train_df.to_csv("train.csv", index=False)
    val_df.to_csv("val.csv", index=False)
    print(f"Split complete. Train: {len(train_df)}, Val: {len(val_df)}")

create_kth_splits('./kth_actions')

/home/user/ghietha1/Lab-Vision-Systems/assignment3
Split complete. Train: 383, Val: 216


In [13]:
class KTHDataset(Dataset):
    def __init__(self, index_file, seq_len=10):
        self.df = pd.read_csv(index_file)
        self.seq_len = seq_len
        self.label_map = {action: i for i, action in enumerate(self.df['action'].unique())}
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        path, action = self.df.iloc[idx][['path', 'action']]
        cap = cv2.VideoCapture(path)
        frames = []
        while len(frames) < self.seq_len:
            ret, frame = cap.read()
            if not ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frames.append(self.transform(frame))
        cap.release()
        return torch.stack(frames), self.label_map[action]

In [14]:
train_ds = KTHDataset('train.csv')
val_ds = KTHDataset('val.csv')

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4)

print("DataLoaders initialized successfully.")

DataLoaders initialized successfully.
